# RS-DAT: start here

This workspace runs **JupyterLab** with **Dask on Snellius** and a **Prefect** server. Work through this notebook once to connect it to Snellius; after that you only repeat step 1 when your Slurm token expires.

**How it works**

- **JupyterLab** and the **Dask scheduler** run here, on this Research Cloud workspace.
- **Dask workers** run as Slurm jobs on Snellius. The workspace submits and cancels them through the **Slurm REST API**, with your **Slurm token**: no SSH login to Snellius from here.
- The workers connect to the scheduler here. For the other direction, a small **gateway job** on Snellius opens an SSH tunnel back to this workspace. It starts with your first cluster and stops when the last one closes.
- **Prefect** runs here too, at `/prefect/` next to `/jupyter/`. Flows can run their tasks on the Dask workers.
- All Dask traffic is encrypted with this workspace's own certificate.

This notebook is your copy: edit it freely. The original is in `/opt/rsdat/share/RS-DAT-start-here.ipynb`.

## 1. Your Slurm token

The workspace acts on Snellius with a token that you create there.

1. Log in to Snellius (SSH, with your second factor) and run, for a token valid for one day:

   ```
   scontrol token lifespan=86400
   ```

2. Copy the line it prints (`SLURM_JWT=eyJ...`).
3. Run the next cell and paste it. It's stored in `~/.config/rsdat/slurm.env`, readable only by this workspace's JupyterLab user.

In a terminal, `rsdat-slurm-token` does the same, and `rsdat-slurm-token --status` shows when the token expires. New terminals show it too.

In [ ]:
import getpass, subprocess

token = getpass.getpass("Slurm token (the SLURM_JWT=... line): ")
result = subprocess.run(["rsdat-slurm-token"], input=token + "\n", text=True, capture_output=True)
print(result.stdout or result.stderr)
del token

Check that Snellius answers with your token (lists your jobs, probably none yet):

In [ ]:
!rsdat-slurm-token --status
!squeue --user "$(rsdat-slurm-token --user)"

## 2. A Dask cluster on Snellius

`RSDATSLURMCluster` takes its defaults (partition, cores, memory, walltime, worker image) from `/etc/dask/rsdat.yaml`; override them per cluster, e.g. `RSDATSLURMCluster(cores=8, memory="16GiB", walltime="02:00:00")`.

Creating the first cluster submits the gateway job and waits for it, so it takes as long as that job waits in the queue. `scale` then submits worker jobs, which join when they start.

- The gateway job also runs a worker, `rsdat-gateway` (14 threads on Snellius: the job has the 16 cores Snellius charges for anyway, and keeps 2 for the tunnel). So `scale(jobs=2)` gives three workers.
- One cluster at a time per workspace: the scheduler always listens on port 8786. Close a cluster (in the sidebar or a notebook) before creating the next.

The **Dask** sidebar on the left does the same: **New** creates a cluster, **Scale** adds workers, and its dashboard links open through JupyterLab.

In [ ]:
from rsdat_dask import RSDATSLURMCluster
from distributed import Client

cluster = RSDATSLURMCluster()
cluster.scale(jobs=2)
client = Client(cluster)
client.wait_for_workers(2, timeout=900)
client

Where did the work run? These are Snellius compute nodes:

In [ ]:
import socket
sorted(set(client.gather(client.map(lambda _: socket.gethostname(), range(20)))))

Close the cluster when you're done: that cancels its worker jobs, and the gateway job when no other cluster in this kernel uses it. If you forget, or the kernel stops, the workers end within a minute and the gateway within five.

In [ ]:
cluster.close()

## 3. Prefect

The Prefect UI is at **`/prefect/`** on this workspace (the Access menu of the workspace, or replace `/jupyter/...` by `/prefect/` in the address bar). It asks once for a password: in a terminal, `echo $PREFECT_API_AUTH_STRING` prints it (the whole `rsdat:...` string).

Flows run from notebooks report to that server. With `DaskTaskRunner`, their tasks run on the Dask workers on Snellius:

In [ ]:
from prefect import flow, task
from prefect_dask import DaskTaskRunner

cluster = RSDATSLURMCluster()
cluster.scale(jobs=1)

@task
def where(x):
    import socket
    return x, socket.gethostname()

@flow(log_prints=True, task_runner=DaskTaskRunner(address=cluster.scheduler_address))
def hello_snellius(n: int = 4):
    for x, host in [f.result() for f in where.map(range(n))]:
        print(x, "ran on", host)

hello_snellius()
cluster.close()

## When something doesn't work

| What you see | Likely cause | What to do |
| --- | --- | --- |
| `sbatch`/`squeue` errors about authentication | No token, or it expired | Step 1 again; `rsdat-slurm-token --status` |
| `Address already in use` (port 8786) | Another cluster is still running, in the sidebar or another notebook | Close it, or restart that notebook's kernel |
| An error saying Slurm refused the job | Partition, account, cores, memory or walltime not accepted | The settings named in the error (`/etc/dask/rsdat.yaml`, or arguments to `RSDATSLURMCluster`) |
| Creating a cluster takes long | The gateway job waits in the queue | `squeue --user "$(rsdat-slurm-token --user)"` shows its state |
| Workers never join | They can't reach the scheduler, or the job failed | The job logs on Snellius, in `~/.rsdat/dask-logs/` |
| Results don't come back | The gateway job stopped | `dask-gateway-<job id>.log` in the same directory |
| Errors about versions or unpickling | Worker environment differs from this workspace | `client.get_versions(check=True)` |

Each Dask job writes its log on Snellius in `~/.rsdat/dask-logs/` (`dask-gateway-<id>.log`, `dask-worker-<id>.log`).